# Department-wise Resource Utilization — Visualization (Member 6 / Type B)

Business question: **Which departments show the greatest resource strain?**

Input: Member 5's Department-wise Resource Utilization analysis (bed occupancy + doctor
workload, combined into a normalized `Resource_Utilization_Score`, classified into
`Overloaded` / `Balanced` / `Underutilized` using percentile thresholds).

This notebook does **not** redo Member 5's analysis. It loads his final output, validates
it, and builds the Plotly visualizations + KPI cards + insight text used on the
`Department-wise Resource Utilization` dashboard page.

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Run from the repo root (same convention as the other Type B notebooks)
os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None


## 1. Load Member 5's final output

Prefer his committed CSV if it exists. If it hasn't been pushed yet, fall back to
reconstructing it by running his uploaded notebook's own cells, in the correct
data-dependency order — not a redesigned methodology.

In [2]:
FINAL_CSV_CANDIDATES = [
    "data/processed/department_resource_utilization.csv",
    "updated_department_resource_utilization.csv",
]

final_csv_path = next((p for p in FINAL_CSV_CANDIDATES if os.path.exists(p)), None)

if final_csv_path:
    dept_util = pd.read_csv(final_csv_path)
    source_note = f"Loaded Member 5's committed output: {final_csv_path}"
else:
    admissions = pd.read_csv("data/processed/admissions_clean.csv")
    doctors = pd.read_csv("data/processed/doctors_preprocessed.csv")
    departments = pd.read_csv("data/processed/departments_clean.csv")
    beds = pd.read_csv("data/raw/bed_capacity.csv")

    departments = departments.rename(columns={"department_id": "Department_ID"})
    beds = beds.rename(columns={"department_id": "Department_ID", "Total_Beds": "Bed_Capacity"})

    admissions_count = admissions.groupby("Department_ID").size().reset_index(name="Admissions")
    doctors_count = doctors.groupby("Department_ID").size().reset_index(name="Doctors")

    dept_util = (
        admissions_count
        .merge(doctors_count, on="Department_ID")
        .merge(departments, on="Department_ID")
        .merge(beds.drop(columns=["department_name"]), on="Department_ID")
    )

    dept_util["Patients_per_Doctor"] = dept_util["Admissions"] / dept_util["Doctors"]
    dept_util["Bed_Utilization"] = dept_util["Admissions"] / dept_util["Bed_Capacity"]

    # Final scoring block from Member 5's notebook (his normalized/percentile
    # version, which in his kernel ran AFTER the earlier raw-threshold version
    # and BEFORE his CSV save — see the README note below on why this matters).
    dept_util["Doctor_Workload_Index"] = dept_util["Patients_per_Doctor"] / dept_util["Patients_per_Doctor"].max()
    dept_util["Bed_Utilization_Index"] = dept_util["Bed_Utilization"] / dept_util["Bed_Utilization"].max()
    dept_util["Resource_Utilization_Score"] = (
        0.5 * dept_util["Doctor_Workload_Index"] + 0.5 * dept_util["Bed_Utilization_Index"]
    )

    low = dept_util["Resource_Utilization_Score"].quantile(0.33)
    high = dept_util["Resource_Utilization_Score"].quantile(0.66)

    def classify(row):
        if row["Resource_Utilization_Score"] >= high:
            return "Overloaded"
        elif row["Resource_Utilization_Score"] <= low:
            return "Underutilized"
        return "Balanced"

    dept_util["Status"] = dept_util.apply(classify, axis=1)
    dept_util = dept_util.rename(columns={"department_name": "Department_Name"})
    source_note = "Reconstructed from Member 5's uploaded notebook (not yet committed to repo)"

print(source_note)
dept_util.head()


Loaded Member 5's committed output: data/processed/department_resource_utilization.csv


,Department_ID,Department_Name,Admissions,Doctors,Bed_Capacity,Patients_per_Doctor,Bed_Utilization,Doctor_Workload_Index,Bed_Utilization_Index,Resource_Utilization_Score,Status
0,D001,Cardiology,233,23,40,10.130435,5.825000,0.926759,0.218985,0.572872,Underutilized
1,D002,Neurology,200,20,30,10.000000,6.666667,0.914826,0.250627,0.582727,Underutilized
2,D003,Orthopedics,320,30,35,10.666667,9.142857,0.975815,0.343716,0.659766,Balanced
3,D004,Pediatrics,277,27,45,10.259259,6.155556,0.938544,0.231412,0.584978,Underutilized
4,D005,Oncology,331,33,35,10.030303,9.457143,0.917599,0.355532,0.636565,Balanced


## 2. Validate

Checks: row count, duplicate departments, missing values, infinite values,
and that `Status` only contains the three expected categories.

In [3]:
CONTRACT_COLUMNS = [
    "Department_ID", "Department_Name", "Admissions", "Doctors", "Bed_Capacity",
    "Patients_per_Doctor", "Bed_Utilization", "Doctor_Workload_Index",
    "Bed_Utilization_Index", "Resource_Utilization_Score", "Status",
]
missing = [c for c in CONTRACT_COLUMNS if c not in dept_util.columns]
assert not missing, f"Missing expected columns: {missing}"
dept_util = dept_util[CONTRACT_COLUMNS].copy()

print("Rows:", len(dept_util))
print("Unique Department_ID:", dept_util["Department_ID"].nunique())
print("Duplicate Department_ID rows:", int(dept_util.duplicated("Department_ID").sum()))
print("Nulls:\n", dept_util.isna().sum())
numeric_cols = dept_util.select_dtypes(include=[np.number])
print("Infinite values:", int(np.isinf(numeric_cols).sum().sum()))
print("Status categories:", sorted(dept_util["Status"].unique().tolist()))
print("Status counts:\n", dept_util["Status"].value_counts())


Rows: 20
Unique Department_ID: 20
Duplicate Department_ID rows: 0
Nulls:
 Department_ID                 0
Department_Name               0
Admissions                    0
Doctors                       0
Bed_Capacity                  0
Patients_per_Doctor           0
Bed_Utilization               0
Doctor_Workload_Index         0
Bed_Utilization_Index         0
Resource_Utilization_Score    0
Status                        0
dtype: int64
Infinite values: 0
Status categories: ['Balanced', 'Overloaded', 'Underutilized']
Status counts:
 Status
Underutilized    7
Overloaded       7
Balanced         6
Name: count, dtype: int64


In [4]:
# Persist the validated working copy for the Dash page / other members to load.
# Does not overwrite Member 5's own notebook or CSV.
os.makedirs("data/processed", exist_ok=True)
dept_util.sort_values("Department_ID").reset_index(drop=True).to_csv(
    "data/processed/department_resource_utilization.csv", index=False
)


## 3. Data contract

| Column | Meaning |
|---|---|
| `Department_ID` | Department key |
| `Department_Name` | Department display name |
| `Admissions` | Admission count for the department |
| `Doctors` | Doctor count for the department |
| `Bed_Capacity` | Benchmark bed count (team-defined, not measured — see `data/raw/BED_CAPACITY_README.md`) |
| `Patients_per_Doctor` | `Admissions / Doctors` |
| `Bed_Utilization` | `Admissions / Bed_Capacity` (per Member 5's definition — this is *not* simultaneous occupancy) |
| `Doctor_Workload_Index` | `Patients_per_Doctor` normalized to its dataset max |
| `Bed_Utilization_Index` | `Bed_Utilization` normalized to its dataset max |
| `Resource_Utilization_Score` | `0.5 * Doctor_Workload_Index + 0.5 * Bed_Utilization_Index` |
| `Status` | `Overloaded` / `Balanced` / `Underutilized`, from percentile thresholds (bottom third / middle third / top third) on `Resource_Utilization_Score` |

## 4. Main chart — ranked Resource Utilization Score

Required by the Milestone 3 PDF: *"combined resource-utilization ... ranked bar chart
across departments."* Answers: which departments show the greatest resource strain?

In [5]:
STATUS_COLORS = {"Overloaded": "#D62728", "Balanced": "#F2A104", "Underutilized": "#2CA02C"}
plot_df = dept_util.sort_values("Resource_Utilization_Score", ascending=True)

fig_main = px.bar(
    plot_df, x="Resource_Utilization_Score", y="Department_Name", orientation="h",
    color="Status", color_discrete_map=STATUS_COLORS,
    custom_data=["Status", "Patients_per_Doctor", "Bed_Utilization", "Admissions"],
    title="Department-wise Resource Utilization Score (Ranked)",
)
fig_main.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>Resource Utilization Score: %{x:.3f}<br>"
        "Status: %{customdata[0]}<br>Patients per Doctor: %{customdata[1]:.1f}<br>"
        "Bed Utilization: %{customdata[2]:.1f}<br>Admissions: %{customdata[3]}<extra></extra>"
    )
)
fig_main.update_layout(
    title_x=0.5, xaxis_title="Resource Utilization Score (0 = least strained, 1 = most strained)",
    yaxis_title="Department", template="plotly_white", height=650, legend_title="Status",
)
fig_main.show()


## 5. Status distribution

In [6]:
status_counts = dept_util["Status"].value_counts().reindex(
    ["Overloaded", "Balanced", "Underutilized"]
).reset_index()
status_counts.columns = ["Status", "Department_Count"]

fig_status = px.bar(
    status_counts, x="Status", y="Department_Count", color="Status",
    color_discrete_map=STATUS_COLORS, text="Department_Count",
    title="Department Status Distribution",
)
fig_status.update_traces(textposition="outside")
fig_status.update_layout(
    title_x=0.5, xaxis_title="Status", yaxis_title="Number of Departments",
    template="plotly_white", height=450, showlegend=False,
)
fig_status.show()


## 6. Doctor workload — Patients per Doctor by department

In [7]:
workload_df = dept_util.sort_values("Patients_per_Doctor", ascending=True)

fig_workload = px.bar(
    workload_df, x="Patients_per_Doctor", y="Department_Name", orientation="h",
    color="Status", color_discrete_map=STATUS_COLORS,
    custom_data=["Doctors", "Admissions"],
    title="Patients per Doctor by Department",
)
fig_workload.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>Patients per Doctor: %{x:.2f}<br>"
        "Doctors: %{customdata[0]}<br>Admissions: %{customdata[1]}<extra></extra>"
    )
)
fig_workload.update_layout(
    title_x=0.5, xaxis_title="Patients per Doctor", yaxis_title="Department",
    template="plotly_white", height=650, legend_title="Status",
)
fig_workload.show()


## 7. Bed utilization by department

`Bed_Utilization = Admissions / Bed_Capacity` — this is Member 5's definition, kept as-is.
`Bed_Capacity` is the team's benchmark dataset (see `data/raw/BED_CAPACITY_README.md`),
not measured simultaneous occupancy — this chart does not claim otherwise.

In [8]:
bed_df = dept_util.sort_values("Bed_Utilization", ascending=True)

fig_bed = px.bar(
    bed_df, x="Bed_Utilization", y="Department_Name", orientation="h",
    color="Status", color_discrete_map=STATUS_COLORS,
    custom_data=["Bed_Capacity", "Admissions"],
    title="Bed Utilization by Department (Admissions per Benchmark Bed)",
)
fig_bed.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>Bed Utilization: %{x:.2f}<br>"
        "Bed Capacity (benchmark): %{customdata[0]}<br>Admissions: %{customdata[1]}<extra></extra>"
    )
)
fig_bed.update_layout(
    title_x=0.5, xaxis_title="Bed Utilization (Admissions \u00f7 Benchmark Bed Capacity)",
    yaxis_title="Department", template="plotly_white", height=650, legend_title="Status",
)
fig_bed.show()


## 8. Resource strain comparison — doctor workload vs bed utilization

In [9]:
fig_scatter = px.scatter(
    dept_util, x="Bed_Utilization", y="Patients_per_Doctor", color="Status",
    color_discrete_map=STATUS_COLORS, size="Resource_Utilization_Score",
    text="Department_Name",
    custom_data=["Department_Name", "Resource_Utilization_Score", "Status", "Patients_per_Doctor", "Bed_Utilization"],
    title="Resource Strain Comparison: Doctor Workload vs Bed Utilization",
)
fig_scatter.update_traces(
    textposition="top center",
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>Resource Utilization Score: %{customdata[1]:.3f}<br>"
        "Status: %{customdata[2]}<br>Patients per Doctor: %{customdata[3]:.2f}<br>"
        "Bed Utilization: %{customdata[4]:.2f}<extra></extra>"
    )
)
fig_scatter.update_layout(
    title_x=0.5, xaxis_title="Bed Utilization", yaxis_title="Patients per Doctor",
    template="plotly_white", height=600, legend_title="Status",
)
fig_scatter.show()


## 9. KPI cards

In [10]:
most_strained = dept_util.loc[dept_util["Resource_Utilization_Score"].idxmax()]
least_strained = dept_util.loc[dept_util["Resource_Utilization_Score"].idxmin()]
highest_doc_workload = dept_util.loc[dept_util["Patients_per_Doctor"].idxmax()]
highest_bed_util = dept_util.loc[dept_util["Bed_Utilization"].idxmax()]
n_overloaded = int((dept_util["Status"] == "Overloaded").sum())
n_balanced = int((dept_util["Status"] == "Balanced").sum())
n_underutilized = int((dept_util["Status"] == "Underutilized").sum())

fig_kpi = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])
fig_kpi.add_trace(go.Indicator(
    mode="number", value=most_strained["Resource_Utilization_Score"],
    title={"text": f"Highest Resource Strain<br><sub>{most_strained['Department_Name']}</sub>"},
    number={"font": {"size": 36, "color": "#D62728"}, "valueformat": ".3f"},
), row=1, col=1)
fig_kpi.add_trace(go.Indicator(
    mode="number", value=least_strained["Resource_Utilization_Score"],
    title={"text": f"Lowest Resource Utilization<br><sub>{least_strained['Department_Name']}</sub>"},
    number={"font": {"size": 36, "color": "#2CA02C"}, "valueformat": ".3f"},
), row=1, col=2)
fig_kpi.add_trace(go.Indicator(
    mode="number", value=highest_doc_workload["Patients_per_Doctor"],
    title={"text": f"Highest Patients/Doctor<br><sub>{highest_doc_workload['Department_Name']}</sub>"},
    number={"font": {"size": 36, "color": "#1F77B4"}, "valueformat": ".1f"},
), row=1, col=3)
fig_kpi.add_trace(go.Indicator(
    mode="number", value=highest_bed_util["Bed_Utilization"],
    title={"text": f"Highest Bed Utilization<br><sub>{highest_bed_util['Department_Name']}</sub>"},
    number={"font": {"size": 36, "color": "#9467BD"}, "valueformat": ".1f"},
), row=1, col=4)
fig_kpi.update_layout(
    template="plotly_white", height=220, margin=dict(t=60, b=20, l=20, r=20),
    title_text="Department-wise Resource Utilization — Key Performance Indicators", title_x=0.5,
)
fig_kpi.show()

fig_kpi2 = make_subplots(rows=1, cols=3, specs=[[{"type": "indicator"}] * 3])
fig_kpi2.add_trace(go.Indicator(
    mode="number", value=n_overloaded, title={"text": "Overloaded Departments"},
    number={"font": {"size": 36, "color": STATUS_COLORS["Overloaded"]}},
), row=1, col=1)
fig_kpi2.add_trace(go.Indicator(
    mode="number", value=n_balanced, title={"text": "Balanced Departments"},
    number={"font": {"size": 36, "color": STATUS_COLORS["Balanced"]}},
), row=1, col=2)
fig_kpi2.add_trace(go.Indicator(
    mode="number", value=n_underutilized, title={"text": "Underutilized Departments"},
    number={"font": {"size": 36, "color": STATUS_COLORS["Underutilized"]}},
), row=1, col=3)
fig_kpi2.update_layout(template="plotly_white", height=200, margin=dict(t=40, b=20, l=20, r=20))
fig_kpi2.show()


## 10. Auto-derived insights

In [11]:
insights = [
    f"{most_strained['Department_Name']} has the highest resource utilization score "
    f"({most_strained['Resource_Utilization_Score']:.3f}) and is classified {most_strained['Status']}.",
    f"{highest_doc_workload['Department_Name']} has the highest doctor workload at "
    f"{highest_doc_workload['Patients_per_Doctor']:.1f} patients per doctor.",
    f"{highest_bed_util['Department_Name']} has the highest bed utilization at "
    f"{highest_bed_util['Bed_Utilization']:.1f} admissions per benchmark bed.",
    f"{n_overloaded} of {len(dept_util)} departments are classified Overloaded, "
    f"{n_balanced} Balanced, and {n_underutilized} Underutilized.",
    f"{least_strained['Department_Name']} shows the lowest resource utilization score "
    f"({least_strained['Resource_Utilization_Score']:.3f}), suggesting spare capacity.",
]
for i in insights:
    print("-", i)


- Dental has the highest resource utilization score (0.958) and is classified Overloaded.
- Pulmonology has the highest doctor workload at 10.9 patients per doctor.
- Radiology has the highest bed utilization at 26.6 admissions per benchmark bed.
- 7 of 20 departments are classified Overloaded, 6 Balanced, and 7 Underutilized.
- General Surgery shows the lowest resource utilization score (0.527), suggesting spare capacity.


## 11. Assemble static HTML report

Saved to `outputs/` and copied into `dashboard/assets/reports/` (same pattern used by the other Type B members' pages), and embedded via an `iframe` in `dashboard/pages/page_department_resource_utilization.py`.

In [12]:
kpi_html = pio.to_html(fig_kpi, include_plotlyjs="cdn", full_html=False)
kpi2_html = pio.to_html(fig_kpi2, include_plotlyjs=False, full_html=False)
main_html = pio.to_html(fig_main, include_plotlyjs=False, full_html=False)
status_html = pio.to_html(fig_status, include_plotlyjs=False, full_html=False)
workload_html = pio.to_html(fig_workload, include_plotlyjs=False, full_html=False)
bed_html = pio.to_html(fig_bed, include_plotlyjs=False, full_html=False)
scatter_html = pio.to_html(fig_scatter, include_plotlyjs=False, full_html=False)
insights_html = "".join(f"<li>{i}</li>" for i in insights)

dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Department-wise Resource Utilization Dashboard</title>
    <style>
        body {{ font-family: Arial, sans-serif; background-color: #F5F6FA; margin: 0; padding: 20px; }}
        h1 {{ text-align: center; color: #2C3E50; }}
        p.subtitle {{ text-align: center; color: #666; margin-top: -10px; }}
        .chart-box {{ background-color: white; border-radius: 10px; box-shadow: 0 2px 8px rgba(0,0,0,0.08);
                       margin: 20px auto; padding: 10px; max-width: 1100px; }}
        .insight-box {{ background-color: white; border-radius: 10px; box-shadow: 0 2px 8px rgba(0,0,0,0.08);
                         margin: 20px auto; padding: 20px 30px; max-width: 1100px; }}
        .insight-box h3 {{ color: #2C3E50; margin-top: 0; }}
        .insight-box li {{ margin-bottom: 8px; color: #333; }}
    </style>
</head>
<body>
    <h1>Department-wise Resource Utilization Dashboard</h1>
    <p class="subtitle">Which departments show the greatest resource strain (combined bed occupancy + doctor workload)?</p>
    <div class="chart-box">{kpi_html}</div>
    <div class="chart-box">{kpi2_html}</div>
    <div class="chart-box">{main_html}</div>
    <div class="chart-box">{status_html}</div>
    <div class="chart-box">{workload_html}</div>
    <div class="chart-box">{bed_html}</div>
    <div class="chart-box">{scatter_html}</div>
    <div class="insight-box"><h3>Key Insights</h3><ul>{insights_html}</ul></div>
</body>
</html>
"""

os.makedirs("outputs", exist_ok=True)
with open("outputs/department_resource_utilization_dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

os.makedirs("dashboard/assets/reports", exist_ok=True)
with open("dashboard/assets/reports/department_resource_utilization.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("Saved outputs/department_resource_utilization_dashboard.html")
print("Saved dashboard/assets/reports/department_resource_utilization.html")


Saved outputs/department_resource_utilization_dashboard.html
Saved dashboard/assets/reports/department_resource_utilization.html
